# 0. Подготовка

In [1]:
import os
# from pathlib import Path
# import shutil
import json
import requests
import re
import pandas as pd
import numpy as np


# Константы
MYINDIE_JAM_URL = "https://myindie.ru/jams/jam/myindie-game-jam-level-9/games?page="
MYINDIE_JAM_URL_PAGES = 3  # Количество страниц с играми на геймджеме
OUTPUT_DIR = "output/"
CRITERIAS = ["art", "sound", "theme", "gameplay", "narrative", "overall_impression"]  # Список критериев для анализа

# Директория для джема
jam_path = re.search(r'https://myindie.ru/jams/jam/([^<]+)/games', MYINDIE_JAM_URL)
if jam_path is None:
    raise ValueError("Не удалось извлечь путь джема из URL. Проверьте правильность MYINDIE_JAM_URL.")
jam_path = str(jam_path.group(1))
os.makedirs(os.path.join(OUTPUT_DIR, jam_path), exist_ok=True)

pd.set_option('display.float_format', '{:.3f}'.format)

# 1. Скачиваем все страницы игр геймджема

1.1 Ищем все игры джема

In [3]:
def get_jam_games_urls(jam_url):
    """Скачивает страницу джема и извлекает URL игр."""

    response = requests.get(jam_url)

    if response.status_code != 200:
        print(f"Ошибка при получении страницы джема: `{response.status_code}`")
        return []

    games_urls = re.findall(r'/games/game/[\w-]+', response.text)
    games_urls = [f"https://myindie.ru{url}" for url in games_urls]
    print(f"Найдено {len(games_urls)} URL игр в: {jam_url}")

    return games_urls


games_urls = []
for page in range(1, MYINDIE_JAM_URL_PAGES + 1):
    paged_url = f"{MYINDIE_JAM_URL}{page}"
    games_urls.extend(get_jam_games_urls(paged_url))

print(f"\nНайдено {len(games_urls)} игр на {MYINDIE_JAM_URL_PAGES} страницах:\n{chr(10).join(games_urls)}")


Найдено 30 URL игр в: https://myindie.ru/jams/jam/myindie-game-jam-level-9/games?page=1
Найдено 30 URL игр в: https://myindie.ru/jams/jam/myindie-game-jam-level-9/games?page=2
Найдено 8 URL игр в: https://myindie.ru/jams/jam/myindie-game-jam-level-9/games?page=3

Найдено 68 игр на 3 страницах:
https://myindie.ru/games/game/ultimatum-odna-noch-s-karachunom
https://myindie.ru/games/game/trash-under-ground
https://myindie.ru/games/game/cult-indie
https://myindie.ru/games/game/udivitelnyj-ded
https://myindie.ru/games/game/franshiza-ktulhu_nx3
https://myindie.ru/games/game/durdom
https://myindie.ru/games/game/exorcismo
https://myindie.ru/games/game/5-days-with-the-necronomicon_opw
https://myindie.ru/games/game/full-moon-twin-rite
https://myindie.ru/games/game/zayachij-kult
https://myindie.ru/games/game/no-edward
https://myindie.ru/games/game/protokol-pentagrammy
https://myindie.ru/games/game/all-hail-sister
https://myindie.ru/games/game/s-run_5pm
https://myindie.ru/games/game/les
https://my

1.2 Скачать все HTML-страницы игр геймджема

In [21]:
jam_games_file_path = os.path.join(OUTPUT_DIR, jam_path, f"jam_games.txt")
if os.path.exists(jam_games_file_path):
    os.remove(jam_games_file_path)
jam_games_file = open(jam_games_file_path, 'a', encoding='utf-8')

# DEBUG_GAMES_MAX = 2  # DEBUG delete
for i, game_url in enumerate(games_urls): #[:DEBUG_GAMES_MAX]):
    print(f"Обработка URL игры: {game_url}")
    response = requests.get(game_url)
    if response.status_code != 200:
        print(f"* Ошибка при получении страницы игры: `{response.status_code}`")
        continue

    title_match = re.search(r'<title>([^<]+)</title>', response.text)
    if title_match:
        title = f"{i:03d}_" + re.sub(r'[^\w_]', '-', re.sub(r'\s+', '_', title_match.group(1).strip()))
    else:
        title = f"{i:03d}_" + "Unknown"

    output_file_path = os.path.join(OUTPUT_DIR, jam_path, f"{title}.html")
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write(response.text)
        print(f"Сохранено в `{output_file_path}`")
    jam_games_file.write(f"{output_file_path} {game_url}\n")

jam_games_file.close()

Обработка URL игры: https://myindie.ru/games/game/ultimatum-odna-noch-s-karachunom
Сохранено в `output/000_УЛЬТИМАТУМ--_Одна_ночь_с_Карачуном-_Жанр-_Horror_-_Инди-игры_-_MyIndie.html`
Обработка URL игры: https://myindie.ru/games/game/trash-under-ground
Сохранено в `output/001_Fetidity-_Жанр-_-_Инди-игры_-_MyIndie.html`
Обработка URL игры: https://myindie.ru/games/game/cult-indie
Сохранено в `output/002_Cult_Indie-_Жанр-_-_Инди-игры_-_MyIndie.html`
Обработка URL игры: https://myindie.ru/games/game/udivitelnyj-ded
Сохранено в `output/003_Удивительный_Дед-_Жанр-_Platformer_-_Инди-игры_-_MyIndie.html`
Обработка URL игры: https://myindie.ru/games/game/franshiza-ktulhu_nx3
Сохранено в `output/004_Франшиза_Ктулху_-_Жанр-_-_Инди-игры_-_MyIndie.html`
Обработка URL игры: https://myindie.ru/games/game/durdom
Сохранено в `output/005_Durdom-_Жанр-_Shooter_-_Инди-игры_-_MyIndie.html`
Обработка URL игры: https://myindie.ru/games/game/exorcismo
Сохранено в `output/006_Exorcismo-_Жанр-_Shooter-_Surviva

# 2. Извлекаем данные со всех страниц игр
2.1 Используем уже скачанные HTML-страницы игр геймджема, чтобы извлечь данные о каждой игре

In [4]:
def unflatten_nuxt_data(data):
    """ Разворачивает плоскую структуру данных Nuxt.js в дерево. """
    if not isinstance(data, list) or not data: return data
    memo = {}
    def resolve(val):
        if isinstance(val, int) and 0 <= val < len(data):
            if val not in memo:
                memo[val] = resolve_item(data[val])
            return memo[val]
        return val
    def resolve_item(item):
        if isinstance(item, dict):
            return {k: resolve(v) for k, v in item.items()}
        if isinstance(item, list):
            return [resolve(v) for v in item]
        return item
    return resolve_item(data[1])

jam_games_file_path = os.path.join(OUTPUT_DIR, jam_path, "jam_games.txt")
jam_games_file = open(jam_games_file_path, 'r', encoding='utf-8')
print(f"Всего игр в файле `{jam_games_file_path}`: {len(jam_games_file.readlines())}")

Всего игр в файле `output/myindie-game-jam-level-9\jam_games.txt`: 68


2.2 Собираем судейские оценки и отзывы для каждой игры из файлов

In [5]:
all_judges_reviews = []

jam_games_file.seek(0)
for line in jam_games_file:
    output_file_path, game_url = line.strip().split(' ', 1)
    print(f"Обработка: {game_url}")
    with open(output_file_path, 'r', encoding='utf-8') as f:
        html_content = f.read()

    match = re.search(r'id=\"__NUXT_DATA__\">([^<]+)</script>', html_content)
    if not match:
        print(f"* Не найден __NUXT_DATA__ в `{output_file_path}`\n")
        continue

    json_data = json.loads(match.group(1))
    unflattened = unflatten_nuxt_data(json_data)
    if not unflattened:
        print(f"* Не удалось развернуть данные Nuxt.js в `{output_file_path}`\n")
        continue

    data_section = unflattened.get('data', [])
    if not data_section:
        print(f"* Не найден раздел 'data' в развернутых данных Nuxt.js в `{output_file_path}`\n")
        continue

    # Берем второй элемент списка (индекс 1) — там словарь с результатами
    if isinstance(data_section, list) and len(data_section) > 1:
        payload_container = data_section[1]
        if not payload_container:
            print(f"* Пустой контейнер в `{output_file_path}`\n")
            continue

        # В словаре берем первый ключ (game<alias>)
        if isinstance(payload_container, dict) and payload_container:
            first_key = list(payload_container.keys())[0]
            game_payload = payload_container[first_key]
            if not game_payload:
                print(f"* Пустой объект игры в `{output_file_path}`\n")
                continue

            # Извлекаем отзывы из game_payload['data']['reviews']
            if isinstance(game_payload, dict):
                inner_data = game_payload.get('data', {})
                if not inner_data:
                    print(f"* Пустой объект 'data' в `{output_file_path}`\n")
                    continue

                reviews = inner_data.get('reviews', [])
                if not reviews:
                    print(f"* Не найдено отзывов судей в `{output_file_path}`\n")
                    continue

                judges = [r for r in reviews if isinstance(r, dict) and r.get('reviewerType') == 'judge']
                all_judges_reviews.extend(judges)
                print(f"Судейских отзывов: {len(judges)}\n")

print(f"Всего судейских отзывов на джеме: {len(all_judges_reviews)}")
jam_games_file.close()

Обработка: https://myindie.ru/games/game/ultimatum-odna-noch-s-karachunom
Судейских отзывов: 3

Обработка: https://myindie.ru/games/game/trash-under-ground
Судейских отзывов: 0

Обработка: https://myindie.ru/games/game/cult-indie
Судейских отзывов: 4

Обработка: https://myindie.ru/games/game/udivitelnyj-ded
Судейских отзывов: 1

Обработка: https://myindie.ru/games/game/franshiza-ktulhu_nx3
Судейских отзывов: 2

Обработка: https://myindie.ru/games/game/durdom
* Не найдено отзывов судей в `output/myindie-game-jam-level-9\005_Durdom-_Жанр-_Shooter_-_Инди-игры_-_MyIndie.html`

Обработка: https://myindie.ru/games/game/exorcismo
Судейских отзывов: 0

Обработка: https://myindie.ru/games/game/5-days-with-the-necronomicon_opw
Судейских отзывов: 3

Обработка: https://myindie.ru/games/game/full-moon-twin-rite
Судейских отзывов: 0

Обработка: https://myindie.ru/games/game/zayachij-kult
Судейских отзывов: 3

Обработка: https://myindie.ru/games/game/no-edward
Судейских отзывов: 4

Обработка: https:/

# 3. Обработка данных и анализ

In [56]:
# print(json.dumps(all_judges_reviews, ensure_ascii=False, indent=2))

3.1 Обрабатываем все судейские отзывы и оценки для каждой игры

In [6]:
# Словари для хранения данных судей
judges_dict = {} # userId -> {"username": username, "reviews": []}

for review in all_judges_reviews:
    if review.get('reviewerType') == 'judge':
        user_id = review.get('userId')
        username = review.get('user', {}).get('username', 'Unknown')

        if user_id not in judges_dict:
            judges_dict[user_id] = {
                "username": username,
                "reviews": []
            }

        judges_dict[user_id]["reviews"].append(review)

print(f"Количество уникальных судей: {len(judges_dict)}")
for user_id, info in judges_dict.items():
    print(f"Судья: {info['username']} (ID: {user_id}) - Обзоров: {len(info['reviews'])}")

# Финальный словарь с идентификацией по username
judges_reviews_by_username = {info['username']: info['reviews'] for info in judges_dict.values()}

Количество уникальных судей: 4
Судья: PoliKhai (ID: c739f699-9424-4039-9c0b-8b26a6f0f1c2) - Обзоров: 41
Судья: Saley (ID: 6ddb6287-691d-41f2-a7db-5df11f84c1cf) - Обзоров: 32
Судья: DmitriySklyarov (ID: 9b681ea6-e7cb-489e-bae7-0e07607f93d4) - Обзоров: 33
Судья: HelenAllienPoe (ID: db51c67e-9d3c-4e25-a589-ec045a0cdb62) - Обзоров: 14


In [ ]:
# print(json.dumps(judges_reviews_by_username, ensure_ascii=False, indent=2))

3.2 Собираем статистику по каждому судье: средний балл, количество оценок, количество отзывов

In [7]:
judges_stats = []

for username, reviews in judges_reviews_by_username.items():
    if not reviews:
        print(f"* Судья {username} не имеет обзоров, пропускаем.")
        continue

    totals = { 'score': [] }
    for criteria in CRITERIAS:
        totals[criteria] = []

    for r in reviews:
        if 'score' in r:
            totals['score'].append(r['score'])

        c = r.get('criterias', {})
        for criteria in CRITERIAS:
            if criteria in c:
                totals[criteria].append(c[criteria])

    judge_row = {
        'username': username,
        'reviews_count': len(reviews)
    }

    for key, values in totals.items():
        judge_row[f'avg_{key}'] = round(sum(values) / len(values), 3) if values else 0

    judges_stats.append(judge_row)

df_judges_stats = pd.DataFrame(judges_stats)

# Вывод результата, отсортированного по среднему баллу
df_judges_stats.sort_values(by='avg_score', ascending=False)


,username,reviews_count,avg_score,avg_art,avg_sound,avg_theme,avg_gameplay,avg_narrative,avg_overall_impression
3,HelenAllienPoe,14,3.829,4.071,4.143,4.143,3.536,3.357,3.679
1,Saley,32,3.528,4.094,3.562,3.422,3.031,3.344,3.609
0,PoliKhai,41,3.305,3.451,3.207,3.695,2.829,3.305,3.268
2,DmitriySklyarov,33,3.115,3.788,3.212,2.833,2.788,2.970,3.015


3.3 Cтатистический анализ: средние оценки, медианы, стандартные отклонения, распределения оценок и длины отзывов

In [8]:
detailed_stats = []

for username, reviews in judges_reviews_by_username.items():
    precise_scores = []
    review_lengths = []

    for r in reviews:
        c = r.get('criterias', {})
        crit_values = [c[k] for k in CRITERIAS if k in c]

        if crit_values:
            game_score = sum(crit_values) / len(crit_values)
            precise_scores.append(game_score)

        text = r.get('reviewText', '')
        if text:
            clean_text = re.sub(r'<[^>]+>', '', text)
            review_lengths.append(len(clean_text))

    stats = {
        'username': username,
        'min_score': np.min(precise_scores) if precise_scores else 0.0,
        'max_score': np.max(precise_scores) if precise_scores else 0.0,
        'median_score': np.median(precise_scores) if precise_scores else 0.0,
        'std_score': np.std(precise_scores) if precise_scores else 0.0,
        'avg_review_chars': np.mean(review_lengths) if review_lengths else 0.0
    }
    detailed_stats.append(stats)

df_detailed = pd.DataFrame(detailed_stats)

df_final_stats = pd.merge(df_judges_stats, df_detailed, on='username')

cols = ['username', 'reviews_count', 'avg_score', 'median_score', 'std_score', 'min_score', 'max_score', 'avg_review_chars']
df_final_stats[cols].sort_values(by='avg_score', ascending=False)


,username,reviews_count,avg_score,median_score,std_score,min_score,max_score,avg_review_chars
3,HelenAllienPoe,14,3.829,3.917,0.485,2.917,4.583,572.643
1,Saley,32,3.528,3.500,0.671,2.083,4.667,777.625
0,PoliKhai,41,3.305,3.417,0.783,1.083,4.667,532.829
2,DmitriySklyarov,33,3.115,3.083,0.666,1.500,4.250,591.455


3.4 Больше статистики судей:
- strictness_index: Отрицательный = строгий, Положительный = добрый

In [11]:
from datetime import datetime

deep_insights = []

all_precise_scores = []
for reviews in judges_reviews_by_username.values():
    for r in reviews:
        c = r.get('criterias', {})
        crit_values = [c[k] for k in CRITERIAS if k in c]
        if crit_values:
            all_precise_scores.append(sum(crit_values) / len(crit_values))

global_avg = np.mean(all_precise_scores) if all_precise_scores else 0.0

for username, reviews in judges_reviews_by_username.items():
    if not reviews: continue

    precise_scores = []
    review_hours = []
    review_dates = []

    # Статистика по критериям для этого судьи
    criteria_totals = {k: [] for k in CRITERIAS}

    for r in reviews:
        c = r.get('criterias', {})
        cv = [c[k] for k in CRITERIAS if k in c]
        if cv:
            precise_scores.append(sum(cv)/len(cv))

        for k in CRITERIAS:
            if k in c: criteria_totals[k].append(c[k])

        # Время
        dt_str = r.get('createdAt')
        if dt_str:
            dt = datetime.strptime(dt_str, "%Y-%m-%dT%H:%M:%S.%fZ")
            review_hours.append(dt.hour)
            review_dates.append(dt.date())

    # любимый и нелюбимый критерий (avg)
    crit_avgs = {k: np.mean(v) if v else 0 for k, v in criteria_totals.items()}
    best_crit = max(crit_avgs, key=crit_avgs.get)
    worst_crit = min(crit_avgs, key=crit_avgs.get)

    judge_avg = np.mean(precise_scores) if precise_scores else 0.0

    insight = {
        'username': username,
        'strictness_index': round(judge_avg - global_avg, 3), # Отрицательный = строгий, Положительный = добрый
        'top_criteria': f"{best_crit} ({round(crit_avgs[best_crit], 2)})",
        'bottom_criteria': f"{worst_crit} ({round(crit_avgs[worst_crit], 2)})",
        'active_days': (max(review_dates) - min(review_dates)).days + 1 if review_dates else 0,
        'peak_hour_utc': max(set(review_hours), key=review_hours.count) if review_hours else 0,
        'reviews_per_day': round(len(reviews) / ((max(review_dates) - min(review_dates)).days + 1), 2) if review_dates else 0
    }
    deep_insights.append(insight)

df_insights = pd.DataFrame(deep_insights)

df_insights.sort_values(by='strictness_index')


,username,strictness_index,top_criteria,bottom_criteria,active_days,peak_hour_utc,reviews_per_day
2,DmitriySklyarov,-0.259,art (3.79),gameplay (2.79),7,19,4.710
0,PoliKhai,-0.067,theme (3.7),gameplay (2.83),3,13,13.670
1,Saley,0.151,art (4.09),gameplay (3.03),3,20,10.670
3,HelenAllienPoe,0.462,sound (4.14),narrative (3.36),4,5,3.500
